# Sector-Sorted Commodity Universe - Build Pipeline
## Xavier Jaeger · Bachelor Thesis · Erasmus University Rotterdam

Constructs the three blocks of the sector-sorted commodity test-asset universe.

| Block | Assets | N |
|---|---|---|
| A - Commodity futures | 28 individual contracts (no sorting) | 28 |
| B - Commodity equity sectors | 4 producer + 8 processor sectors (WRDS/CRSP) | 12 |
| C - Commodity currencies | 12 individual currency pairs | 12 |

**Total: N = 52, T = 223 months (Feb 1997 – Aug 2015)**

Run order: Cell 0 → Cell 1 (Block A) → Cell 2 (Block C) → Cell 3 (Block B, WRDS) → Cell 4 (merge)

Cell 3 requires WRDS. If unavailable, skip it and place a pre-built
`equity_sector_monthly.csv` in `Output/`.


In [1]:
import numpy as np
import pandas as pd
import os, warnings
warnings.filterwarnings('ignore')

# File paths
PATH_FUTURES    = 'Input/commodity_futures_data.xlsx'
PATH_CURRENCIES = 'Input/commodity_currencies_data.xlsx'
PATH_ADBEAR     = 'Input/ADBear_Monthly_Formation.csv'
PATH_MST_DATES  = 'Input/mExcess_Returns_Farago_Tedongap_201508_LM.csv'
WRDS_USER       = 'fdxavierj5'

# Sample window
SAMPLE_START = '1997-02-01'
SAMPLE_END   = '2015-08-31'
STUDY_START  = '1995-01-04'
STUDY_END    = '2023-12-31'

# Constants
MIN_FIRMS_PER_SECTOR = 10
RUB_WINSORISE        = True
MIN_DAYS_PER_WEEK    = 3

os.makedirs('Output', exist_ok=True)
print('Configuration loaded.')

Configuration loaded.


---
## Cell 1 - Block A: Individual Commodity Futures (N=28)

Each of the 28 commodity futures contracts is its own portfolio.
RBOB Gasoline is excluded (only 51% coverage over 1996-2015).
Returns: TRc1 (nearby) settlement prices, log-compounded weekly Wed-to-Wed,
then within-month compounded to monthly.

In [2]:
CONTRACT_MAP = {
    'NYM-LIGHT CRUDE OIL TRc1 - SETT. PRICE'       : 'F_WTI_CRUDE',
    'ICE-BRENT CRUDE OIL TRc1 - SETT. PRICE'       : 'F_BRENT_CRUDE',
    'NYM-NATURAL GAS TRc1 - SETT. PRICE'           : 'F_NAT_GAS',
    'NYM-NY HARBOR ULSD TRc1 - SETT. PRICE'        : 'F_HEATING_OIL',
    'ICE-GAS OIL TRc1 - SETT. PRICE'               : 'F_GAS_OIL',
    'CMX-GOLD 100 OZ TRc1 - SETT. PRICE'           : 'F_GOLD',
    'CMX-SILVER 5000 OZ TRc1 - SETT. PRICE'        : 'F_SILVER',
    'NYM-PALLADIUM TRc1 - SETT. PRICE'             : 'F_PALLADIUM',
    'NYM-PLATINUM TRc1 - SETT. PRICE'              : 'F_PLATINUM',
    'CMX-HIGH GRADE COPPER TRc1 - SETT. PRICE'     : 'F_COPPER',
    'LME-ALUMINIUM TRc1 - SETT. PRICE'             : 'F_ALUMINIUM',
    'LME-ZINC TRc1 - SETT. PRICE'                  : 'F_ZINC',
    'LME-NICKEL TRc1 - SETT. PRICE'                : 'F_NICKEL',
    'LME-LEAD TRc1 - SETT. PRICE'                  : 'F_LEAD',
    'LME-TIN TRc1 - SETT. PRICE'                   : 'F_TIN',
    'CBT-CORN COMP. TRc1 - SETT. PRICE'            : 'F_CORN',
    'CBT-WHEAT COMPOSITE TRc1 - SETT. PRICE'       : 'F_WHEAT_CBOT',
    'KCBT-HRW WHEAT COMP TRc1 - SETT. PRICE'       : 'F_WHEAT_KCBT',
    'CBT-SOYBEANS COMP. TRc1 - SETT. PRICE'        : 'F_SOYBEANS',
    'CSCE-COTTON #2 TRc1 - SETT. PRICE'            : 'F_COTTON',
    'CSCE-SUGAR #11 TRc1 - SETT. PRICE'            : 'F_SUGAR',
    "CSCE-COFFEE 'C' TRc1 - SETT. PRICE"           : 'F_COFFEE',
    'CSCE-COCOA TRc1 - SETT. PRICE'                : 'F_COCOA',
    'NYCE-(FCOJ-A) ORANGE JUICE TRc1 - SETT. PRICE': 'F_ORANGE_JUICE',
    'CME-LUMBER TRc1 - SETT. PRICE'                : 'F_LUMBER',
    'CME-LIVE CATTLE COMP. TRc1 - SETT. PRICE'     : 'F_LIVE_CATTLE',
    'CME-FEEDER CATTLE COMP. TRc1 - SETT. PRICE'   : 'F_FEEDER_CATTLE',
    'CME-LEAN HOGS COMP. TRc1 - SETT. PRICE'       : 'F_LEAN_HOGS',
}

raw = pd.read_excel(PATH_FUTURES, sheet_name=0, header=None)
name_row = raw.iloc[3, :]
col_for  = {name_row[c]: c for c in range(1, raw.shape[1])
            if isinstance(name_row[c], str)}
body      = raw.iloc[6:, :].copy()
dates_raw = pd.to_datetime(body.iloc[:, 0], errors='coerce')
body      = body[dates_raw.notna()]
dates_raw = dates_raw[dates_raw.notna()]
body.index = dates_raw

px = {}
for long_name, ticker in CONTRACT_MAP.items():
    if long_name not in col_for:
        print(f'  [WARN] not found: {long_name}'); continue
    s = pd.to_numeric(body.iloc[:, col_for[long_name]], errors='coerce')
    px[ticker] = s.replace(0.0, np.nan)
px = pd.DataFrame(px).sort_index().loc[STUDY_START:STUDY_END]

daily_ret  = px.pct_change().clip(-0.5, 0.5)
log_daily  = np.log1p(daily_ret)
week_id    = daily_ret.index.to_period('W-WED')
log_wk     = log_daily.groupby(week_id).sum(min_count=1)
n_valid    = daily_ret.notna().groupby(week_id).sum()
fut_weekly = np.expm1(log_wk)
fut_weekly[n_valid < MIN_DAYS_PER_WEEK] = np.nan
fut_weekly.index = fut_weekly.index.to_timestamp(how='end').normalize()
fut_weekly.index.name = 'date'

fut_monthly = (1 + fut_weekly).resample('ME').prod() - 1
n_wk        = fut_weekly.notna().resample('ME').sum()
fut_monthly = fut_monthly.where(n_wk > 0)
fut_monthly.index = fut_monthly.index.to_period('M').to_timestamp('M')

print(f'Block A: {fut_monthly.shape[1]} futures contracts')
cov = fut_weekly.notna().mean().sort_values()
print(f'  Weekly coverage: min={cov.min():.3f} ({cov.index[0]}), median={cov.median():.3f}')
print(f'  Annualised mean (full period): {fut_monthly.mean().mean()*12*100:.2f}% pa')
fut_monthly.to_csv('Output/block_A_futures_monthly.csv')
print('Saved Output/block_A_futures_monthly.csv')

Block A: 28 futures contracts
  Weekly coverage: min=0.999 (F_WTI_CRUDE), median=0.999
  Annualised mean (full period): 7.94% pa
Saved Output/block_A_futures_monthly.csv


---
## Cell 2 - Block C: Individual Currency Pairs (N=12)

All 12 commodity-exporter currencies kept as individual portfolios -
no commodity-beta sorting. Positive return = FCY appreciation vs USD.
RUB winsorised at [0.5%, 99.5%] to address the 1998 devaluation outlier
(see sector_robustness.py diagnostic D8).

In [3]:
CURRENCY_NAMES = {
    'AUSTDO$':'C_AUD','CNDOLL$':'C_CAD','NZDOLL$':'C_NZD','NORKRO$':'C_NOK',
    'COMRAN$':'C_ZAR','BRACRU$':'C_BRL','CHILPE$':'C_CLP','CISRUB$':'C_RUB',
    'MEXPES$':'C_MXN','PERUSO$':'C_PEN','MALADL$':'C_MYR','INDORU$':'C_IDR',
}
INVERT = {'CNDOLL$','NORKRO$','COMRAN$','BRACRU$','CHILPE$',
          'CISRUB$','MEXPES$','PERUSO$','MALADL$','INDORU$'}

meta     = pd.read_excel(PATH_CURRENCIES, header=None, nrows=6, dtype=str)
code_row = meta.iloc[4, :]
codes    = [v for v in code_row.dropna().tolist() if v != 'Code']
raw_cur  = pd.read_excel(PATH_CURRENCIES, header=None, skiprows=6,
                          usecols=range(len(codes)+1), index_col=0)
raw_cur.index   = pd.to_datetime(raw_cur.index, errors='coerce')
raw_cur.columns = codes
raw_cur = raw_cur[raw_cur.index.notna()].sort_index()
raw_cur = raw_cur.apply(pd.to_numeric, errors='coerce')

fx = raw_cur.copy()
for code in codes:
    if code in INVERT:
        fx[code] = 1.0 / fx[code]
fx = fx.rename(columns=CURRENCY_NAMES)[list(CURRENCY_NAMES.values())]
fx = fx.loc[STUDY_START:STUDY_END]

log_d  = np.log(fx).diff()
wk_id  = fx.index.to_period('W-WED')
log_wk = log_d.groupby(wk_id).sum(min_count=1)
cur_weekly = np.expm1(log_wk)
cur_weekly.index = cur_weekly.index.to_timestamp(how='end').normalize()
cur_weekly.index.name = 'date'

if RUB_WINSORISE:
    rub = cur_weekly['C_RUB'].dropna()
    lo, hi = rub.quantile(0.005), rub.quantile(0.995)
    n_clip = ((cur_weekly['C_RUB'] < lo) | (cur_weekly['C_RUB'] > hi)).sum()
    cur_weekly['C_RUB'] = cur_weekly['C_RUB'].clip(lo, hi)
    print(f'RUB winsorised at [{lo:.4f}, {hi:.4f}], {n_clip} obs clipped')

cur_monthly = (1 + cur_weekly).resample('ME').prod() - 1
n_wk2       = cur_weekly.notna().resample('ME').sum()
cur_monthly = cur_monthly.where(n_wk2 > 0)
cur_monthly.index = cur_monthly.index.to_period('M').to_timestamp('M')

print(f'Block C: {cur_monthly.shape[1]} currency pairs')
cov2 = cur_weekly.notna().mean().sort_values()
print(f'  Weekly coverage: min={cov2.min():.3f} ({cov2.index[0]}), median={cov2.median():.3f}')
cur_monthly.to_csv('Output/block_C_currencies_monthly.csv')
print('Saved Output/block_C_currencies_monthly.csv')

RUB winsorised at [-0.1473, 0.0906], 16 obs clipped
Block C: 12 currency pairs
  Weekly coverage: min=0.959 (C_RUB), median=0.999
Saved Output/block_C_currencies_monthly.csv


---
## Cell 3 - Block B: Commodity Equity Sectors (WRDS/CRSP)

12 value-weighted sector portfolios: 4 producer sectors (long commodity
price) and 8 processor sectors (short commodity price). Common shares only
on NYSE/AMEX/Nasdaq. (Originally scoped as 8 producer + 9 processor
categories; several were merged or dropped during SIC-range assignment --
see PRODUCER_SIC/PROCESSOR_SIC below for what's actually implemented.)
Sector-months with fewer than 10 firms are set to NaN (Issue 8 mitigation).

**Skip this cell if you already have `Output/block_B_equity_monthly.csv`.**


In [9]:
PRODUCER_SIC = {
    'EQ_P_FossilFuels' : [(1200,1299),(1300,1339),(1380,1389)],
    'EQ_P_Mining'      : [(1000,1039),(1040,1049),(1050,1099),(1400,1499)],
    'EQ_P_AgForestry'  : [(100,299),(700,799),(800,899),(2400,2439)],
    'EQ_P_NatGasUtil'  : [(4920,4939)],
}
PROCESSOR_SIC = {
    'EQ_X_Refining'      : [(2900,2999)],
    'EQ_X_Chemicals'     : [(2800,2899)],
    'EQ_X_RubberPlastic' : [(3000,3099),(3100,3199)],
    'EQ_X_SteelPrimary'  : [(3300,3329),(3390,3399)],
    'EQ_X_AlumFabMetal'  : [(3330,3389),(3400,3499)],
    'EQ_X_FoodProcessing': [(2000,2009),(2040,2048),(2060,2079),(2090,2099)],
    'EQ_X_MeatDairy'     : [(2010,2039),(2050,2059)],
    'EQ_X_PaperWood'     : [(2440,2499),(2600,2699)],
}
SECTOR_SIC = {**PRODUCER_SIC, **PROCESSOR_SIC}
SIC_TO_SECTOR = {}
for sector, ranges in SECTOR_SIC.items():
    for lo_s, hi_s in ranges:
        for sic in range(lo_s, hi_s+1):
            SIC_TO_SECTOR[sic] = sector

all_sics = sorted(SIC_TO_SECTOR.keys())
sic_list = ','.join(str(s) for s in all_sics)

import wrds
db = wrds.Connection(wrds_username=WRDS_USER)
print('Connected to WRDS.')

sql = (
    "SELECT a.permno, a.date, a.ret, a.prc, a.shrout, b.siccd, b.shrcd, b.exchcd "
    "FROM crsp.msf AS a "
    "LEFT JOIN crsp.msenames AS b ON a.permno=b.permno "
    "AND b.namedt<=a.date AND a.date<=b.nameendt "
    "WHERE a.date BETWEEN '1994-07-01' AND '2023-12-31' "
    "AND b.shrcd IN (10,11) AND b.exchcd IN (1,2,3) "
    f"AND b.siccd IN ({sic_list})"
)
msf = db.raw_sql(sql, date_cols=['date'])
db.close()
print(f'Pulled {len(msf):,} firm-month obs')

msf = msf.dropna(subset=['ret','siccd']).copy()
msf['siccd']  = msf['siccd'].astype(int)
msf['sector'] = msf['siccd'].map(SIC_TO_SECTOR)
msf = msf.dropna(subset=['sector'])
msf['mktcap'] = msf['prc'].abs() * msf['shrout']
msf = msf.sort_values(['permno','date'])
msf['w'] = msf.groupby('permno')['mktcap'].shift(1)
msf = msf.dropna(subset=['w'])
msf = msf[msf['w'] > 0]

Loading library list...
Done
Connected to WRDS.
Pulled 308,280 firm-month obs
Block B: 12 equity sector portfolios
Median/min firm counts per sector:
  EQ_P_FossilFuels         median=138  min=83
  EQ_P_Mining              median=33  min=16
  EQ_P_AgForestry          median=15  min=8
  EQ_P_NatGasUtil          median=51  min=32
  EQ_X_Refining            median=19  min=10
  EQ_X_Chemicals           median=344  min=192
  EQ_X_RubberPlastic       median=37  min=25
  EQ_X_SteelPrimary        median=25  min=16
  EQ_X_AlumFabMetal        median=65  min=44
  EQ_X_FoodProcessing      median=32  min=21
  EQ_X_MeatDairy           median=31  min=17
  EQ_X_PaperWood           median=36  min=18
Saved Output/block_B_equity_monthly.csv and block_B_firmcounts.csv


In [ ]:
def vw_ret(g):
    return np.average(g['ret'], weights=g['w'])
grp   = msf.groupby(['date','sector'])
vwret = grp.apply(vw_ret).unstack('sector')
nfirm = grp.size().unstack('sector')
vwret = vwret.where(nfirm >= MIN_FIRMS_PER_SECTOR)
MIN_FIRMS_AG = 5
for col in ['EQ_P_AgForestry']:
    if col in vwret.columns:
        vwret[col] = vwret[col].where(nfirm[col] >= MIN_FIRMS_AG)
vwret.index = pd.to_datetime(vwret.index)
nfirm.index = pd.to_datetime(nfirm.index)

ordered = [c for c in list(PRODUCER_SIC)+list(PROCESSOR_SIC) if c in vwret.columns]
eq_monthly = vwret[ordered].copy()
eq_monthly.index = eq_monthly.index.to_period('M').to_timestamp('M')
nfirm.index = nfirm.index.to_period('M').to_timestamp('M')

print(f'Block B: {eq_monthly.shape[1]} equity sector portfolios')
print('Median/min firm counts per sector:')
for c in ordered:
    print(f'  {c:<24} median={nfirm[c].median():.0f}  min={nfirm[c].min():.0f}')

eq_monthly.to_csv('Output/block_B_equity_monthly.csv')
nfirm[ordered].to_csv('Output/block_B_firmcounts.csv')
print('Saved Output/block_B_equity_monthly.csv and block_B_firmcounts.csv')

---
## Cell 4 - Merge blocks and align to ADBear window

In [ ]:
fut_m = pd.read_csv('Output/block_A_futures_monthly.csv',   index_col=0, parse_dates=True)
eq_m  = pd.read_csv('Output/block_B_equity_monthly.csv',    index_col=0, parse_dates=True)
cur_m = pd.read_csv('Output/block_C_currencies_monthly.csv',index_col=0, parse_dates=True)

for df in [fut_m, eq_m, cur_m]:
    df.index = df.index.to_period('M').to_timestamp('M')

uni = fut_m.join(eq_m, how='inner').join(cur_m, how='inner')
uni = uni.loc[SAMPLE_START:SAMPLE_END]

# Handle residual missing obs: <=15 months (~1 year) fill forward/back; drop otherwise
for c in list(uni.columns):
    n_miss = uni[c].isna().sum()
    if   n_miss == 0: continue
    elif n_miss <= 15: uni[c] = uni[c].ffill().bfill()
    else:
        print(f'  Dropping {c}: {n_miss} missing obs'); uni = uni.drop(columns=c)

# Build ADBear aligned to this monthly calendar
adb_raw   = pd.read_csv(PATH_ADBEAR)
mst_raw   = pd.read_csv(PATH_MST_DATES)
mst_dates = pd.to_datetime(mst_raw['Date'].astype(str), format='%Y%m', errors='coerce')
if mst_dates.isna().all():
    mst_dates = pd.to_datetime(mst_raw['Date'].astype(str), errors='coerce')
adb_series = pd.Series(
    adb_raw['ADBearRF_Average_within_the_Month'].values[:len(mst_dates)],
    index=mst_dates).dropna()
adb_series.index = adb_series.index.to_period('M').to_timestamp('M')
adbear = adb_series.reindex(uni.index, method='ffill')

block_map = {}
for c in uni.columns:
    if   c.startswith('F_'):    block_map[c] = 'Futures'
    elif c.startswith('EQ_P_'): block_map[c] = 'EquityProducer'
    elif c.startswith('EQ_X_'): block_map[c] = 'EquityProcessor'
    elif c.startswith('C_'):    block_map[c] = 'Currency'

from collections import Counter
T_uni, N_uni = uni.shape
print(f'Sector universe: T={T_uni} months, N={N_uni}')
print(f'  Window: {uni.index.min().date()} to {uni.index.max().date()}')
print(f'  Blocks: {dict(Counter(block_map.values()))}')
print(f'  ADBear non-missing: {adbear.notna().sum()}/{T_uni}')

# Verify ADBear alignment against thesis values
for theta_chk in [0.125, 0.0]:
    T_D_chk = (adbear >= theta_chk).sum()
    print(f'  theta={theta_chk}: T_D={T_D_chk} ({T_D_chk/T_uni*100:.1f}%)')

uni.to_csv('Output/sector_test_asset_universe.csv')
adbear.to_frame('ADBear').to_csv('Output/sector_adbear_monthly.csv')
pd.Series(block_map, name='block').to_csv('Output/sector_block_map.csv')
print()
print('Outputs saved to Output/:')
print('  sector_test_asset_universe.csv  - main return matrix')
print('  sector_adbear_monthly.csv       - ADBear aligned to this calendar')
print('  sector_block_map.csv            - block labels per column')


Sector universe: T=223 months, N=52
  Window: 1997-02-28 to 2015-08-31
  Blocks: {'Futures': 28, 'EquityProducer': 4, 'EquityProcessor': 8, 'Currency': 12}
  ADBear non-missing: 223/223
  theta=0.125: T_D=74 (33.2%)
  theta=0.0: T_D=75 (33.6%)

Outputs saved to Output/:
  sector_test_asset_universe.csv  — main return matrix
  sector_adbear_monthly.csv       — ADBear aligned to this calendar
  sector_block_map.csv            — block labels per column
